In [28]:
# pip install lonboard geopandas matplotlib pyarrow folium mapclassify scikit-learn

In [29]:
import geopandas as gpd
import numpy as np
from pathlib import Path
from lonboard import viz
import requests

DATASET_DIR = Path("../dataset")
TILFLUKTSROM_DATASET = DATASET_DIR / "tilfluktsrom.gpkg"
BEFOLKNING_DATASET = DATASET_DIR / "Befolkning_42_Agder_25832_BefolkningsstatistikkRutenett1km2025_GML.gpkg"

tilfluktsrom_gdf = gpd.read_file(TILFLUKTSROM_DATASET)
befolkning_gdf = gpd.read_file(BEFOLKNING_DATASET)

viz(befolkning_gdf.to_crs("EPSG:4326"))

# curl -X 'GET' \
#   'https://api.kartverket.no/kommuneinfo/v1/fylker/42' \
#   -H 'accept: application/json'

AGDER_URL = "https://api.kartverket.no/kommuneinfo/v1/fylker/42"
AGDER_POLYGON = requests.get(AGDER_URL).json()["avgrensningsboks"]["coordinates"][0]

/home/user/git/uia/IS-218/.venv/lib/python3.14/site-packages/pyogrio/geopandas.py:382: UserWarning: More than one layer found in 'Befolkning_42_Agder_25832_BefolkningsstatistikkRutenett1km2025_GML.gpkg': 'BefolkningPåRuter1km' (default), 'BefolkningPåRuter1kmGrense'. Specify layer parameter to avoid this warning.
  result = read_func(


In [ ]:
import pandas as pd
from shapely.geometry import Polygon

buffer_lengde = 500
tilfluktsrom_buffer_gdf = tilfluktsrom_gdf.buffer(500)

shelter_gdf = tilfluktsrom_gdf.to_crs("EPSG:25832").copy()
pop_gdf = befolkning_gdf.to_crs("EPSG:25832").copy()

tilfluktsrom_buffer_gdf = shelter_gdf.copy()
tilfluktsrom_buffer_gdf["geometry"] = shelter_gdf.geometry.buffer(buffer_lengde)

pop_in_buffer = gpd.sjoin(pop_gdf, tilfluktsrom_buffer_gdf, how="inner", predicate="overlaps")
agder_polygon_gdf = gpd.GeoDataFrame(
    {"name": ["Agder"]},
    geometry=[Polygon(AGDER_POLYGON)],
    crs="EPSG:4326",
)

import folium
m = folium.Map(location=[58.0, 8.0], zoom_start=7) 
folium.GeoJson(agder_polygon_gdf.to_crs("EPSG:4326")).add_to(m)

m

/run/user/1001/ipykernel_91216/414889086.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  tilfluktsrom_buffer_gdf = tilfluktsrom_gdf.buffer(500)
